# Traffic Sign Recognition: from training to deployment 🚦
**Dataset:** GTSRB (German Traffic Sign Recognition Benchmark): 43 classes, ~39,000 images (26,640 train + 12,630 test in the torchvision version).

**Goal:** not only a good model, but a model that is **small, fast and runs without PyTorch**, like in a car.

| Part | What | Topic |
|---|---|---|
| 1 | Load and explore the data | Python / CV |
| 2 | Leak-free validation split (by track) | ML theory |
| 3 | Load all images into memory (fast training) | PyTorch |
| 4 | Data augmentation (and why **no flips**!) | Computer Vision |
| 5 | A small CNN built for deployment | PyTorch |
| 6 | Training | PyTorch |
| 7 | Evaluation on the official test set | ML |
| 8 | Deployment: size, speed, ONNX, INT8 quantization, a standalone script | Deployment |

# Part 1 – Load and explore the data

In [ ]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# AMD GPU (ROCm) fix: MIOpen fails to compile its BatchNorm kernel on this PC.
# This makes PyTorch use its own built-in GPU kernels instead.
torch.backends.cudnn.enabled = False

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

data_dir = Path("data/gtsrb/GTSRB")

In [ ]:
CLASS_NAMES = [
    "Speed limit 20", "Speed limit 30", "Speed limit 50", "Speed limit 60", "Speed limit 70",
    "Speed limit 80", "End of speed limit 80", "Speed limit 100", "Speed limit 120", "No passing",
    "No passing >3.5t", "Right-of-way at intersection", "Priority road", "Yield", "Stop",
    "No vehicles", "Vehicles >3.5t prohibited", "No entry", "General caution", "Dangerous curve left",
    "Dangerous curve right", "Double curve", "Bumpy road", "Slippery road", "Road narrows on the right",
    "Road work", "Traffic signals", "Pedestrians", "Children crossing", "Bicycles crossing",
    "Beware of ice/snow", "Wild animals crossing", "End of all limits", "Turn right ahead", "Turn left ahead",
    "Ahead only", "Go straight or right", "Go straight or left", "Keep right", "Keep left",
    "Roundabout mandatory", "End of no passing", "End of no passing >3.5t",
]
NUM_CLASSES = len(CLASS_NAMES)   # 43

# --- Training images: folder name = class, file name = "<track>_<photo>.ppm"
train_paths = sorted((data_dir / "Training").glob("*/*.ppm"))
train_labels = np.array([int(p.parent.name) for p in train_paths])          # "00014" -> 14
train_tracks = np.array([f"{p.parent.name}_{p.stem.split('_')[0]}" for p in train_paths])  # unique per real sign

# --- Test images: labels are in a CSV file (columns separated by ";")
test_df = pd.read_csv("data/gtsrb/GT-final_test.csv", sep=";")
test_paths = [data_dir / "Final_Test" / "Images" / f for f in test_df["Filename"]]
test_labels = test_df["ClassId"].to_numpy()

print(f"train images: {len(train_paths)}   real signs (tracks): {len(set(train_tracks))}")
print(f"test images:  {len(test_paths)}")
test_df.head()

In [ ]:
# How many images per class? (an IMBALANCED dataset)
counts = np.bincount(train_labels, minlength=NUM_CLASSES)

plt.figure(figsize=(14, 4))
plt.bar(range(NUM_CLASSES), counts)
plt.xticks(range(NUM_CLASSES))
plt.xlabel("class id")
plt.ylabel("train images")
plt.title(f"Images per class: min {counts.min()} ({CLASS_NAMES[counts.argmin()]}), "
          f"max {counts.max()} ({CLASS_NAMES[counts.argmax()]})")
plt.show()

In [ ]:
# One example image per class
fig, axes = plt.subplots(5, 9, figsize=(15, 9))
for class_id, ax in enumerate(axes.flat):
    ax.axis("off")
    if class_id >= NUM_CLASSES:
        continue
    first = np.where(train_labels == class_id)[0][15]     # the 15th photo: closer = a bit bigger
    ax.imshow(Image.open(train_paths[first]))
    ax.set_title(f"{class_id}: {CLASS_NAMES[class_id][:18]}", fontsize=8)
plt.tight_layout()
plt.show()

sizes = np.array([Image.open(p).size for p in train_paths[::20]])   # (width, height) of every 20th image
print(f"image width:  min {sizes[:, 0].min()}  median {np.median(sizes[:, 0]):.0f}  max {sizes[:, 0].max()}")

# Part 2 – A leak-free validation split
GTSRB was filmed from a driving car. Every real sign appears in a **track of 30 photos**, taken one after another, so they are nearly identical.

⚠️ A random split puts photo #5 in train and photo #6 in validation → the model "remembers" instead of "recognizes" → fake high validation accuracy.

✅ We split by **track**: all 30 photos of one sign go to the same side.

In [ ]:
rng = np.random.default_rng(42)
val_tracks = set()
for class_id in range(NUM_CLASSES):
    tracks = np.unique(train_tracks[train_labels == class_id])     # the real signs of this class
    n_val = max(1, int(round(0.2 * len(tracks))))                  # 20% of them, at least 1
    val_tracks.update(rng.choice(tracks, size=n_val, replace=False))

is_val = np.array([t in val_tracks for t in train_tracks])
tr_idx, val_idx = np.where(~is_val)[0], np.where(is_val)[0]

print(f"train: {len(tr_idx)} images   val: {len(val_idx)} images   ({len(val_tracks)} tracks)")
shared = set(train_tracks[tr_idx]) & set(train_tracks[val_idx])
print(f"tracks in BOTH sets: {len(shared)}  (must be 0)")

# Part 3 – Load all images into memory
Opening 26,000 image files from disk **every epoch** is slow. The images are tiny, so we load them **once**, resize to 32×32 and keep them in one tensor.

Memory: 39,270 images × 3 × 32 × 32 bytes ≈ **120 MB** (as `uint8`, 0-255). Easy.

We also save this tensor to a cache file, so the next time it loads in 1 second.

In [ ]:
IMG_SIZE = 32


def load_images(paths):
    """List of image files -> uint8 tensor [N, 3, 32, 32]."""
    images = np.zeros((len(paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        images[i] = np.asarray(img)                       # shape: [32, 32, 3]  (height, width, color)
    return torch.from_numpy(images).permute(0, 3, 1, 2).contiguous()   # -> [N, 3, 32, 32] (PyTorch order)


cache_file = Path(f"data/gtsrb/cache_{IMG_SIZE}px.pt")
if cache_file.exists():
    cache = torch.load(cache_file)
else:
    start = time.time()
    cache = {"train": load_images(train_paths), "test": load_images(test_paths)}
    torch.save(cache, cache_file)
    print(f"loaded images from disk in {time.time() - start:.1f} s")

all_train_images, test_images = cache["train"], cache["test"]
print(f"all_train_images: {all_train_images.shape} {all_train_images.dtype}")   # [26640, 3, 32, 32] uint8
print(f"test_images:      {test_images.shape}")                                  # [12630, 3, 32, 32]

# Part 4 – Data augmentation (and why NO flips!)
Real-life photos of signs are a little rotated, shifted, far away, dark or bright. We simulate that.

⛔ **No horizontal flip!** A flipped "Keep right" *is* a "Keep left". A flipped "Turn left" becomes "Turn right". The flip would **change the label** → we would teach the model wrong answers. (Unlike the casting photos, where a flip changed nothing.)

**Normalization:** we subtract the mean color and divide by the std (computed on **train** images only), so the inputs are centered around 0. This makes training more stable.

In [ ]:
# mean and std per color channel, from TRAIN images only
train_float = all_train_images[tr_idx].float() / 255
MEAN = train_float.mean(dim=(0, 2, 3)).tolist()   # average over images, height, width -> 3 numbers
STD = train_float.std(dim=(0, 2, 3)).tolist()
del train_float
print(f"MEAN: {[round(m, 3) for m in MEAN]}   STD: {[round(s, 3) for s in STD]}")

train_transform = v2.Compose([
    v2.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # small rotation / shift / zoom
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),          # light conditions
    v2.ToDtype(torch.float32, scale=True),                                 # uint8 0-255 -> float 0-1
    v2.Normalize(MEAN, STD),
])
eval_transform = v2.Compose([                                              # NO randomness for val/test
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])


class SignDataset(Dataset):
    """Images already in memory -> (transformed image, label)."""
    def __init__(self, images, labels, transform):
        self.images = images          # uint8 [N, 3, 32, 32]
        self.labels = torch.tensor(labels)   # tensor() makes a copy (pandas arrays can be read-only)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]


train_ds = SignDataset(all_train_images[tr_idx], train_labels[tr_idx], train_transform)
val_ds = SignDataset(all_train_images[val_idx], train_labels[val_idx], eval_transform)
test_ds = SignDataset(test_images, test_labels, eval_transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)
test_loader = DataLoader(test_ds, batch_size=512)

x, y = next(iter(train_loader))
print(f"one batch: x {x.shape} {x.dtype}, y {y.shape}")   # x [128, 3, 32, 32] float32, y [128]

In [ ]:
# See the augmentation: one image, 8 random versions
img = all_train_images[tr_idx[100]]
plt.figure(figsize=(14, 2.5))
plt.subplot(1, 9, 1)
plt.imshow(img.permute(1, 2, 0))                     # [3, 32, 32] -> [32, 32, 3] for matplotlib
plt.title("original")
plt.axis("off")
for i in range(8):
    aug = v2.Compose(train_transform.transforms[:2])(img)   # only the random part, still uint8
    plt.subplot(1, 9, i + 2)
    plt.imshow(aug.permute(1, 2, 0))
    plt.axis("off")
plt.show()

# Part 5 – A small CNN built for deployment
In a car, the model runs on a small computer next to many other programs. So we want **few parameters** (small file, little memory) and **few calculations** (fast).

Design: 3 blocks of (Conv → BN → ReLU → Conv → BN → ReLU → MaxPool), then **global average pooling** instead of a big Linear layer.

Shapes: `[B,3,32,32] → [B,32,16,16] → [B,64,8,8] → [B,128,4,4] → [B,128] → [B,43]`

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),  # no bias: BatchNorm adds its own
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2),                                                 # halves height and width
    )


class TrafficSignNet(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # [B, 32, 16, 16]
            conv_block(32, 64),    # [B, 64, 8, 8]
            conv_block(64, 128),   # [B, 128, 4, 4]
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # average each feature map: [B, 128, 4, 4] -> [B, 128, 1, 1]
            nn.Flatten(),              # [B, 128]
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = TrafficSignNet(NUM_CLASSES).to(device)
print(model(x.to(device)).shape)                                      # smoke test -> [128, 43]
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}  ->  about {n_params * 4 / 1024:.0f} KB as float32")

# Part 6 – Training
- **CrossEntropyLoss with label smoothing (0.1):** the target is "90% sure" instead of "100% sure". The model becomes less over-confident, which often generalizes better.
- **Cosine learning-rate schedule:** start with big steps, end with tiny steps to fine-tune.
- We keep the model with the **best validation accuracy**.

In [ ]:
import copy


def run_epoch(model, loader, loss_fn, optimizer=None):
    """Train if an optimizer is given, otherwise just evaluate. Returns (loss, accuracy %)."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            count += len(y)
    return total_loss / count, correct / count * 100


EPOCHS = 15
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, best_state = 0, None
start = time.time()

for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, loss_fn, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader, loss_fn)
    scheduler.step()                                     # lower the learning rate a bit
    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)
    if va_acc > best_val_acc:
        best_val_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch+1:2d} | train loss {tr_loss:.3f} acc {tr_acc:6.2f}% "
          f"| val loss {va_loss:.3f} acc {va_acc:6.2f}% | {time.time() - start:5.0f} s")

model.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train"); axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.show()

# Part 7 – Evaluation on the official test set
The test set comes from **different tracks** recorded separately. It's the real exam. (For reference: humans score about 98.8% on GTSRB.)

In [ ]:
def predict_all(model, loader):
    model.eval()
    preds = []
    with torch.inference_mode():
        for X, _ in loader:
            preds.append(model(X.to(device)).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()


test_pred = predict_all(model, test_loader)
test_acc = (test_pred == test_labels).mean() * 100
print(f"Test accuracy: {test_acc:.2f}%   ({(test_pred != test_labels).sum()} mistakes out of {len(test_labels)})")

# Accuracy per class -> which signs are hardest?
per_class = np.array([(test_pred[test_labels == c] == c).mean() * 100 for c in range(NUM_CLASSES)])
worst = np.argsort(per_class)[:8]
print("\nHardest classes:")
for c in worst:
    wrong = test_pred[(test_labels == c) & (test_pred != c)]
    most_confused = CLASS_NAMES[np.bincount(wrong).argmax()] if len(wrong) else "-"
    print(f"  {c:2d} {CLASS_NAMES[c]:30s} {per_class[c]:6.2f}%   most often confused with: {most_confused}")

In [ ]:
# Look at some mistakes
wrong_idx = np.where(test_pred != test_labels)[0]
show = np.random.default_rng(0).choice(wrong_idx, size=min(12, len(wrong_idx)), replace=False)

plt.figure(figsize=(15, 6))
for i, idx in enumerate(show):
    plt.subplot(2, 6, i + 1)
    plt.imshow(test_images[idx].permute(1, 2, 0))       # [3, 32, 32] -> [32, 32, 3]
    plt.title(f"true: {CLASS_NAMES[test_labels[idx]][:16]}\npred: {CLASS_NAMES[test_pred[idx]][:16]}",
              fontsize=8, color="red")
    plt.axis("off")
plt.suptitle("Mistakes on the test set (32x32 pixels: sometimes hard even for humans!)")
plt.tight_layout()
plt.show()

# Part 8 – Deployment 🚗
This is what car makers care about. Four questions:
1. **How big** is the model? (memory on a small chip)
2. **How fast** is it? (a car at 100 km/h drives 28 m per second!)
3. **Can it run without PyTorch?** (PyTorch is huge, embedded devices use small runtimes) → **ONNX**
4. **Can we make it smaller and faster?** → **INT8 quantization**

To simulate a small embedded CPU, we measure on the **CPU with 1 thread** and **1 image at a time** (batch size 1), like a real-time camera.

In [ ]:
deploy_dir = Path("deploy")
deploy_dir.mkdir(exist_ok=True)

model_cpu = copy.deepcopy(model).cpu().eval()
torch.set_num_threads(1)            # simulate a small CPU

torch.save(model_cpu.state_dict(), deploy_dir / "traffic_sign.pth")

# Speed test with PyTorch: one image at a time
one_image = test_ds[0][0].unsqueeze(0)            # [1, 3, 32, 32]
with torch.inference_mode():
    for _ in range(20):                           # warm-up (the first runs are always slower)
        model_cpu(one_image)
    start = time.perf_counter()
    for _ in range(500):
        model_cpu(one_image)
torch_ms = (time.perf_counter() - start) / 500 * 1000

torch_kb = (deploy_dir / "traffic_sign.pth").stat().st_size / 1024
print(f"PyTorch: {torch_kb:.0f} KB   {torch_ms:.3f} ms per image   ({1000 / torch_ms:.0f} images per second)")

## 8.2 Export to ONNX
**ONNX** = a standard file format for neural networks. It's like a **PDF for models**: you make it in one program (PyTorch) and open it anywhere (ONNX Runtime, TensorRT on NVIDIA, OpenVINO on Intel, phones, car chips…).

In [ ]:
import onnxruntime as ort

onnx_path = deploy_dir / "traffic_sign_fp32.onnx"
torch.onnx.export(
    model_cpu,
    (one_image,),                                  # an example input: shows the shape [1, 3, 32, 32]
    str(onnx_path),
    input_names=["image"],
    output_names=["logits"],
    dynamic_shapes={"x": {0: torch.export.Dim("batch")}},   # allow any batch size
    dynamo=True,
    verbose=False,
    external_data=False,       # keep the weights INSIDE the .onnx file -> one file to copy to the device
)


def ort_session(path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1               # 1 thread, like before
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])


sess = ort_session(onnx_path)

# Check: ONNX must give the same output as PyTorch
with torch.inference_mode():
    torch_out = model_cpu(one_image).numpy()
onnx_out = sess.run(None, {"image": one_image.numpy()})[0]
print(f"max difference PyTorch vs ONNX: {np.abs(torch_out - onnx_out).max():.2e}")   # tiny = OK

In [ ]:
# Prepare the whole test set as one numpy array (normalized), for the ONNX tests
X_test_np = torch.stack([test_ds[i][0] for i in range(len(test_ds))]).numpy()   # [12630, 3, 32, 32] float32


def evaluate_onnx(path, name):
    sess = ort_session(path)
    # accuracy (in batches of 512)
    preds = np.concatenate([sess.run(None, {"image": X_test_np[i:i + 512]})[0].argmax(axis=1)
                            for i in range(0, len(X_test_np), 512)])
    acc = (preds == test_labels).mean() * 100
    # speed: one image at a time
    img = X_test_np[:1]
    for _ in range(20):
        sess.run(None, {"image": img})
    start = time.perf_counter()
    for _ in range(500):
        sess.run(None, {"image": img})
    ms = (time.perf_counter() - start) / 500 * 1000
    kb = Path(path).stat().st_size / 1024
    print(f"{name:14s} {kb:7.0f} KB  {ms:7.3f} ms/image  {1000 / ms:7.0f} img/s   test acc {acc:6.2f}%")
    return {"name": name, "kb": kb, "ms": ms, "acc": acc}


results = [{"name": "PyTorch FP32", "kb": torch_kb, "ms": torch_ms, "acc": test_acc}]
results.append(evaluate_onnx(onnx_path, "ONNX FP32"))

## 8.3 INT8 quantization: 4× smaller
Normally every weight is a **32-bit float** (4 bytes). Quantization stores it as an **8-bit integer** (1 byte, values −128…127) plus a scale factor.

📏 Analogy: instead of measuring every distance in millimeters with 7 decimals, you use a ruler with 256 marks. A tiny bit less precise, but 4× less to write down, and integer math is faster on small chips.

**Static quantization** needs **calibration data**: we show it a few hundred training images, so it learns the typical range of the numbers inside the network.

In [ ]:
from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                      quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process


class CalibrationReader(CalibrationDataReader):
    """Gives the quantizer 500 TRAIN images (never test images!), one at a time."""
    def __init__(self, n=500):
        idx = np.random.default_rng(0).choice(len(tr_idx), size=n, replace=False)
        self.images = [val_ds.transform(all_train_images[tr_idx[i]]).unsqueeze(0).numpy() for i in idx]
        self.pos = 0

    def get_next(self):
        if self.pos == len(self.images):
            return None                               # None = "no more data"
        self.pos += 1
        return {"image": self.images[self.pos - 1]}


prep_path = deploy_dir / "traffic_sign_prep.onnx"
int8_path = deploy_dir / "traffic_sign_int8.onnx"
quant_pre_process(str(onnx_path), str(prep_path))     # clean up + fuse Conv and BatchNorm first
quantize_static(
    str(prep_path), str(int8_path), CalibrationReader(),
    quant_format=QuantFormat.QDQ,
    per_channel=True,                                 # one scale per filter -> better accuracy
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
)
prep_path.unlink()                                    # we don't need the in-between file

results.append(evaluate_onnx(int8_path, "ONNX INT8"))

In [ ]:
summary = pd.DataFrame(results).set_index("name").round(2)
summary.columns = ["size (KB)", "ms / image", "test acc (%)"]
summary["images / s"] = (1000 / summary["ms / image"]).round(0)
print(summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, col in zip(axes, ["size (KB)", "ms / image", "test acc (%)"]):
    ax.bar(summary.index, summary[col], color=["tab:gray", "tab:blue", "tab:green"])
    ax.set_title(col)
axes[2].set_ylim(summary["test acc (%)"].min() - 2, 100)
plt.tight_layout()
plt.show()

## 8.4 A standalone app (no PyTorch needed!)
For the device we need 3 things in the `deploy/` folder:
1. `traffic_sign_int8.onnx`: the model
2. `labels.json`: class names + the normalization numbers (MEAN, STD)
3. `predict_sign.py`: a small script that only needs `onnxruntime`, `numpy` and `Pillow`

⚠️ **Classic deployment bug:** the device must do **exactly the same preprocessing** as training (resize to 32×32, the same MEAN/STD). If not, accuracy drops silently. That's why we save MEAN and STD in a file and don't type them by hand.

In [ ]:
with open(deploy_dir / "labels.json", "w") as f:
    json.dump({"class_names": CLASS_NAMES, "img_size": IMG_SIZE, "mean": MEAN, "std": STD}, f, indent=1)

# Run the standalone script like it would run on the device (a separate Python process)
import subprocess
import sys

for i in [0, 1, 2]:
    out = subprocess.run([sys.executable, "deploy/predict_sign.py", str(test_paths[i])],
                         capture_output=True, text=True)
    print(out.stdout.strip() or out.stderr)
    print(f"   (true label: {CLASS_NAMES[test_labels[i]]})\n")

# Summary 🎤
**What I built**
1. A small CNN (~290k parameters) for 43 traffic sign classes, trained with augmentation (and *no* flips, because they change the meaning of a sign).
2. A **leak-free validation split by track**: 30 photos of one sign never land on both sides.
3. A deployment pipeline: **PyTorch → ONNX → INT8 quantization**, measured on 1 CPU thread with batch size 1, like an embedded device.
4. A **standalone predictor** that runs without PyTorch.

**Honest limits**
- I measured on a laptop CPU with 1 thread. Real embedded hardware (Raspberry Pi, a car ECU) is slower. The next step: copy the `deploy/` folder to a Raspberry Pi and measure there.
- GTSRB images are already **cropped around the sign**. A real car system first needs a **detector** to find the sign in the full camera image (e.g. YOLO), and then this classifier.

**Interview line:**
> "Training to 98% was the easy part. I took the model to deployment: exported it to ONNX, quantized it to INT8 to make it about 3.6× smaller and over 2× faster, checked the accuracy stayed the same, and measured the latency for real-time use."